# Build `fact_sales`

Takes the keyed order data + `dim_customer` (SCD Type 2) produced by `02_build_dimensions.ipynb` and assembles the final fact table at the order-line grain.

Every other dimension key (`product_key`, `location_key`, `order_date_key`, `ship_date_key`) was already resolved in the previous notebook via a simple lookup. `customer_key` is different — it depends on *when* the order happened relative to that customer's SCD2 version history, so it's resolved here.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

In [2]:
orders = pd.read_csv('../data/processed/orders_with_keys.csv')
dim_customer = pd.read_csv('../data/processed/dim_customer.csv')

# Dates get saved as plain strings in CSV, so re-parse them as datetime
orders['Order Date'] = pd.to_datetime(orders['Order Date'])
orders['Ship Date'] = pd.to_datetime(orders['Ship Date'])
dim_customer['effective_date'] = pd.to_datetime(dim_customer['effective_date'])
dim_customer['end_date'] = pd.to_datetime(dim_customer['end_date'])

print(f"orders rows: {orders.shape[0]}")
print(f"dim_customer rows (versions): {dim_customer.shape[0]}")

orders rows: 9800
dim_customer rows (versions): 796


## Resolve `customer_key` against SCD2 date ranges

Most customers have exactly one row in `dim_customer`, so the match is trivial. The 3 customers with simulated history have two rows each — for those, an order line must match whichever version was current on that order's date.

Approach: join on `Customer ID` (this temporarily creates 2 candidate rows for the 3 simulated customers), then keep only the row where the order date actually falls inside that version's `[effective_date, end_date]` window.

In [3]:
# Step 1: join on Customer ID (natural key) — creates duplicate candidate rows
# only for customers with more than one dim_customer version
candidates = orders.merge(
    dim_customer[['Customer ID', 'customer_key', 'effective_date', 'end_date']],
    on='Customer ID',
    how='left'
)

print(f"Rows after join (expect some duplication vs. {orders.shape[0]}): {candidates.shape[0]}")

Rows after join (expect some duplication vs. 9800): 9832


In [4]:
# Step 2: keep only the candidate row whose date window actually contains the order date.
# end_date is NaT for the current version, meaning "open-ended" — treat that as always satisfied.
in_range = (
    (candidates['Order Date'] >= candidates['effective_date']) &
    (candidates['end_date'].isna() | (candidates['Order Date'] <= candidates['end_date']))
)

fact_ready = candidates[in_range].drop(columns=['effective_date', 'end_date'])

print(f"Rows after date-range filter: {fact_ready.shape[0]} (should equal original orders count: {orders.shape[0]})")

Rows after date-range filter: 9800 (should equal original orders count: 9800)


In [5]:
# Sanity checks before trusting this join
assert fact_ready.shape[0] == orders.shape[0], \
    "Row count changed — either an order matched 0 versions or more than 1 version."
assert fact_ready['customer_key'].isnull().sum() == 0, \
    "Some orders failed to match any customer_key."
assert fact_ready['Row ID'].is_unique, \
    "Row ID should still be unique after the join — duplicates mean an order matched multiple customer versions."

print("customer_key resolved cleanly for every order line.")

customer_key resolved cleanly for every order line.


**If any of these assertions fail**, the most likely cause is overlapping or gapped date ranges in `dim_customer` for one of the simulated customers (e.g., `end_date` set before `effective_date`, or a gap between one version's `end_date` and the next version's `effective_date` that swallows an order). Go back to the SCD2 simulation cell in `02_build_dimensions.ipynb` and check the 3 sampled customers' date ranges line up with no gaps and no overlaps.

## Recompute `shipping_delay`

This measure was explored in EDA but never persisted into `orders_with_keys.csv` — deriving it here, right before building the fact table.

In [6]:
fact_ready['shipping_delay'] = (fact_ready['Ship Date'] - fact_ready['Order Date']).dt.days

assert (fact_ready['shipping_delay'] >= 0).all(), "Found a negative shipping delay — Ship Date before Order Date."
print(f"Avg shipping delay: {fact_ready['shipping_delay'].mean():.2f} days")

Avg shipping delay: 3.96 days


## Assemble `fact_sales`

Grain: **one row per order line** (matches `Row ID` in the source data).

- Foreign keys: `customer_key`, `product_key`, `location_key`, `order_date_key`, `ship_date_key`
- Degenerate dimension: `Ship Mode` (only 4 values — kept directly on the fact table rather than given its own dimension, since a separate table would add a join for no real analytical benefit)
- Measures: `Sales`, `shipping_delay`
- Reference fields kept for traceability: `Order ID`, `Row ID` (original source key)

In [7]:
fact_sales = fact_ready[[
    'Row ID',
    'Order ID',
    'customer_key',
    'product_key',
    'location_key',
    'order_date_key',
    'ship_date_key',
    'Ship Mode',
    'Sales',
    'shipping_delay',
]].copy()

fact_sales.insert(0, 'fact_sales_key', fact_sales.index + 1)  # surrogate PK

fact_sales.head()

,fact_sales_key,Row ID,Order ID,customer_key,product_key,location_key,order_date_key,ship_date_key,Ship Mode,Sales,shipping_delay
0,1,1,CA-2017-152156,1,1,1,20171108,20171111,Second Class,261.9600,3
1,2,2,CA-2017-152156,1,2,1,20171108,20171111,Second Class,731.9400,3
2,3,3,CA-2017-138688,2,3,2,20170612,20170616,Second Class,14.6200,4
3,4,4,US-2016-108966,3,4,3,20161011,20161018,Standard Class,957.5775,7
4,5,5,US-2016-108966,3,5,3,20161011,20161018,Standard Class,22.3680,7


## Final validation against EDA baseline numbers

Cross-check the fact table against the counts found during EDA — if these don't match, something was lost or duplicated somewhere in the pipeline.

In [8]:
print(f"fact_sales rows: {fact_sales.shape[0]} (expected: 9800)")
print(f"Distinct customer_key values: {fact_sales['customer_key'].nunique()} (expected: 793 base customers + 3 extra SCD2 versions = 796)")
print(f"Distinct product_key values: {fact_sales['product_key'].nunique()} (expected: up to 1893)")
print(f"Any null foreign keys?\n{fact_sales.isnull().sum()}")
print(f"Total Sales in fact_sales: {fact_sales['Sales'].sum():,.2f}")

fact_sales rows: 9800 (expected: 9800)
Distinct customer_key values: 796 (expected: 793 base customers + 3 extra SCD2 versions = 796)
Distinct product_key values: 1893 (expected: up to 1893)
Any null foreign keys?
fact_sales_key    0
Row ID            0
Order ID          0
customer_key      0
product_key       0
location_key      0
order_date_key    0
ship_date_key     0
Ship Mode         0
Sales             0
shipping_delay    0
dtype: int64
Total Sales in fact_sales: 2,261,536.78


**Insight (fill in after running):** `fact_sales` row count should exactly equal the original 9,800 order lines from EDA — no more, no less. `Total Sales` here should match summing `Sales` directly on the raw CSV; if it doesn't, a join above duplicated or dropped rows silently. This total is also the number to check against later once everything is loaded into Postgres, so you can prove the warehouse reproduces the source data exactly.

## Save `fact_sales`

In [9]:
fact_sales.to_csv('../data/processed/fact_sales.csv', index=False)
print("Saved fact_sales.csv")

Saved fact_sales.csv
